In [ ]:
import networkx as nx
import numpy as np

def k_rank(adjacency_matrix, K, alpha=0.85, mu=-0.6, Nmax=100):
    """
    K-rank algorithm for community detection in a network.
    """
    n = adjacency_matrix.shape[0]  # Obtient le nombre de sommets dans le réseau
    seeds = find_seeds(adjacency_matrix, alpha, mu)  # Trouve les graines initiales
    clusters = [[] for _ in range(K)]  # Initialise une liste de clusters pour stocker les communautés
    for i in range(n):  # Parcourt tous les sommets du réseau
        max_similarity = -np.inf  # Initialise la similarité maximale
        next_seed = None  # Initialise la prochaine graine
        r = signal_propagation(adjacency_matrix, i, alpha, Nmax)  # Propage le signal à partir du sommet actuel
        for j in range(n):  # Parcourt tous les sommets du réseau
            if j not in seeds:  # Vérifie si le sommet n'est pas déjà une graine
                similarity = np.dot(r, signal_propagation(adjacency_matrix, j, alpha, Nmax))  # Calcule la similarité entre les signaux
                if similarity > max_similarity:  # Vérifie si la similarité est supérieure à la similarité maximale actuelle
                    max_similarity = similarity  # Met à jour la similarité maximale
                    next_seed = j  # Met à jour la prochaine graine
        if max_similarity < mu:  # Vérifie si la similarité maximale est inférieure à la valeur seuil mu
            break  # Sort de la boucle si la similarité maximale est inférieure à mu
        seeds.append(next_seed)  # Ajoute la prochaine graine à la liste des graines
    for i in range(n):  # Parcourt à nouveau tous les sommets du réseau
        max_similarity = -np.inf  # Réinitialise la similarité maximale
        best_cluster = None  # Initialise le meilleur cluster
        r = signal_propagation(adjacency_matrix, i, alpha, Nmax)  # Propage le signal à partir du sommet actuel
        for j in range(K):  # Parcourt tous les clusters
            similarity = np.dot(r, signal_propagation(adjacency_matrix, seeds[j], alpha, Nmax))  # Calcule la similarité entre les signaux
            if similarity > max_similarity:  # Vérifie si la similarité est supérieure à la similarité maximale actuelle
                max_similarity = similarity  # Met à jour la similarité maximale
                best_cluster = j  # Met à jour le meilleur cluster
        clusters[best_cluster].append(i)  # Ajoute le sommet au meilleur cluster
    return clusters  # Retourne les clusters contenant les communautés détectées

def find_seeds(adjacency_matrix, alpha, mu):
    """
    Finds initial seeds based on PageRank and signal similarity.
    """
    n = adjacency_matrix.shape[0]  # Obtient le nombre de sommets dans le réseau
    eigvals, eigvecs = np.linalg.eig(adjacency_matrix.toarray().T)  # Calcule les valeurs et vecteurs propres de la matrice d'adjacence transposée
    index = np.argsort(eigvals.real)[-1]  # Trouve l'indice de la plus grande valeur propre
    r = eigvecs[:, index].real  # Obtient le vecteur propre correspondant
    seeds = [np.argmax(r)]  # Initialise la liste des graines avec le sommet ayant la plus grande PageRank
    for i in range(1, n):  # Parcourt tous les sommets du réseau à partir du deuxième
        max_similarity = -np.inf  # Initialise la similarité maximale
        next_seed = None  # Initialise la prochaine graine
        r = signal_propagation(adjacency_matrix, seeds[-1], alpha)  # Propage le signal à partir de la dernière graine sélectionnée
        for j in range(n):  # Parcourt tous les sommets du réseau
            if j not in seeds:  # Vérifie si le sommet n'est pas déjà une graine
                similarity = np.dot(r, signal_propagation(adjacency_matrix, j, alpha))  # Calcule la similarité entre les signaux
                if similarity > max_similarity:  # Vérifie si la similarité est supérieure à la similarité maximale actuelle
                    max_similarity = similarity  # Met à jour la similarité maximale
                    next_seed = j  # Met à jour la prochaine graine
        if max_similarity < mu:  # Vérifie si la similarité maximale est inférieure à la valeur seuil mu
            break  # Sort de la boucle si la similarité maximale est inférieure à mu
        seeds.append(next_seed)  # Ajoute la prochaine graine à la liste des graines
    return seeds  # Retourne la liste des graines initiales

def signal_propagation(adjacency_matrix, source, alpha, Nmax=100):
    """
    Signal propagation to compute vertex similarities.
    """
    n = adjacency_matrix.shape[0]  # Obtient le nombre de sommets dans le réseau
    U = np.linalg.matrix_power(np.eye(n) + alpha * adjacency_matrix.toarray(), Nmax)  # Calcule la matrice de propagation du signal
    return U[source]  # Retourne le signal propagé à partir du sommet source


In [ ]:
import networkx as nx
import numpy as np

# Read the graph from the .gml file
G = nx.read_gml("DataSets/réels/dolphins/dolphins.gml")

# Convert the graph to an adjacency matrix
A = nx.adjacency_matrix(G)
print("Size of adjacency matrix:", A.shape)


# Convert the adjacency matrix to a NumPy array
A_array = A.toarray()

print(A_array)
clusters = k_rank(A, 2)
print(clusters)

Size of adjacency matrix: (62, 62)
[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 1]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 1 ... 0 0 0]]
[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61], []]


In [ ]:
def accuracy(predicted_clusters, ground_truth_labels):
    """
    Compute accuracy given predicted clusters and ground truth labels.
    """
    if len(predicted_clusters) != len(ground_truth_labels):
        raise ValueError("Length of predicted clusters and ground truth labels must be the same.")

    # Flatten the ground truth labels list
    flattened_ground_truth = [label for sublist in ground_truth_labels for label in sublist]

    # Count the number of correctly predicted labels
    correct = sum(1 for pred, truth in zip(predicted_clusters, flattened_ground_truth) if pred == truth)

    # Total number of labels
    total = len(flattened_ground_truth)

    # Compute accuracy
    accuracy = correct / total
    return accuracy

# Example usage:
predicted_clusters_k_rank = [0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0]
ground_truth_labels = [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61], [], []]

acc = accuracy(predicted_clusters_k_rank, ground_truth_labels)
print("Accuracy:", acc)


ValueError: Length of predicted clusters and ground truth labels must be the same.

In [ ]:
adjacency_matrix = np.array([[0, 1, 1, 0, 0],
                             [1, 0, 1, 1, 0],
                             [1, 1, 0, 1, 1],
                             [0, 1, 1, 0, 1],
                             [0, 0, 1, 1, 0]])

k = 2  # Le nombre de communautés à rechercher

# Fonction de similarité (à remplacer par votre propre fonction)
def similarity_function(A, vertex):
    return np.sum(A[vertex])

cluster_assignments = k_rankk(adjacency_matrix, k, similarity_function)  # Corrected by fixing indentation

# Indented code block for further processing of cluster_assignments
print(cluster_assignments)  # Example of processing output



TypeError: 'int' object is not subscriptable

In [ ]:
from sklearn.metrics.cluster import normalized_mutual_info_score

def read_labels_from_file(file_path):
    """
    Read the labels from a text file.
    """
    with open(file_path, 'r') as file:
        labels = [int(line.strip()) for line in file]
    return labels

def compute_accuracy(ground_truth_labels, predicted_clusters):
    """
    Compute accuracy by comparing ground truth labels with predicted clusters.
    """
    # Assuming labels start from 0 and are consecutive integers
    n = len(ground_truth_labels)
    correct = sum(1 for i in range(n) if ground_truth_labels[i] == predicted_clusters[i])
    accuracy = correct / n
    return accuracy

def compute_nmi(ground_truth_labels, predicted_clusters):
    """
    Compute NMI (Normalized Mutual Information) between ground truth labels and predicted clusters.
    """
    nmi = normalized_mutual_info_score(ground_truth_labels, predicted_clusters)
    return nmi

# Example usage:
ground_truth_file = "ground_truth_labels.txt"
predicted_clusters_file = "predicted_clusters.txt"

ground_truth_labels = read_labels_from_file(ground_truth_file)
predicted_clusters = read_labels_from_file(predicted_clusters_file)

accuracy = compute_accuracy(ground_truth_labels, predicted_clusters)
nmi = compute_nmi(ground_truth_labels, predicted_clusters)

print("Accuracy:", accuracy)
print("NMI:", nmi)
